# Google Gemini (google-genai) 사용 예시

> 우리 프로젝트의 멀티AI에서 Gemini를 활용하는 방법
> 
> - 라이브러리: `google-generativeai`
> - 모델: `gemini-2.5-flash` (현재 사용 중)

In [13]:
# pip install google-generativeai python-dotenv
import google.generativeai as genai
import os
from dotenv import load_dotenv

# trading-monitor/.env에서 API 키 로드
load_dotenv('../trading-monitor/.env')

api_key = os.getenv('GEMINI_API_KEY')
if not api_key:
    print("❌ GEMINI_API_KEY를 찾을 수 없습니다")
    print("   trading-monitor/.env 파일을 확인하세요")
else:
    genai.configure(api_key=api_key)
    print(f"✅ Gemini API 설정 완료 (키: ...{api_key[-6:]})")

✅ Gemini API 설정 완료 (키: ...ipsf_k)


## 1. 기본 텍스트 생성

In [14]:
# 사용 가능한 모델 목록 확인
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"  {m.name}")

  models/gemini-2.5-flash
  models/gemini-2.5-pro
  models/gemini-2.0-flash
  models/gemini-2.0-flash-001
  models/gemini-2.0-flash-lite-001
  models/gemini-2.0-flash-lite
  models/gemini-2.5-flash-preview-tts
  models/gemini-2.5-pro-preview-tts
  models/gemma-3-1b-it
  models/gemma-3-4b-it
  models/gemma-3-12b-it
  models/gemma-3-27b-it
  models/gemma-3n-e4b-it
  models/gemma-3n-e2b-it
  models/gemma-4-26b-a4b-it
  models/gemma-4-31b-it
  models/gemini-flash-latest
  models/gemini-flash-lite-latest
  models/gemini-pro-latest
  models/gemini-2.5-flash-lite
  models/gemini-2.5-flash-image
  models/gemini-3-pro-preview
  models/gemini-3-flash-preview
  models/gemini-3.1-pro-preview
  models/gemini-3.1-pro-preview-customtools
  models/gemini-3.1-flash-lite-preview
  models/gemini-3-pro-image-preview
  models/nano-banana-pro-preview
  models/gemini-3.1-flash-image-preview
  models/lyria-3-clip-preview
  models/lyria-3-pro-preview
  models/gemini-robotics-er-1.5-preview
  models/gemini-robo

In [16]:
model = genai.GenerativeModel('gemini-2.5-flash')

# 간단한 질문
response = model.generate_content('AAPL 주식의 현재 투자 전망을 한국어로 3줄로 요약해줘')
print(response.text)

AAPL은 강력한 브랜드 충성도와 서비스 매출 성장을 바탕으로 견고한 펀더멘털을 유지하고 있습니다.
다만 중국 시장 둔화, 규제 리스크, 높은 밸류에이션은 단기적인 변동성 요인으로 작용할 수 있습니다.
장기적으로는 AI 전략과 신제품 혁신이 성장의 핵심이며, 6월 WWDC에서 공개될 AI 비전이 중요합니다.


## 2. 펀더멘탈 데이터 기반 분석 (우리 프로젝트 방식)

In [18]:
import yfinance as yf

# yfinance에서 실데이터 가져오기
stock = yf.Ticker('AAPL')
info = stock.info

# 펀더멘탈 컨텍스트 구성
fundamental_context = f"""
종목: {info.get('longName')} ({info.get('symbol')})
섹터: {info.get('sector')} / {info.get('industry')}

가치 평가:
- PER: {info.get('trailingPE', 'N/A')}
- Forward PER: {info.get('forwardPE', 'N/A')}
- PBR: {info.get('priceToBook', 'N/A')}
- EV/EBITDA: {info.get('enterpriseToEbitda', 'N/A')}

수익성:
- ROE: {info.get('returnOnEquity', 'N/A')}
- ROA: {info.get('returnOnAssets', 'N/A')}
- 영업이익률: {info.get('operatingMargins', 'N/A')}
- 순이익률: {info.get('profitMargins', 'N/A')}

성장성:
- 매출 성장률: {info.get('revenueGrowth', 'N/A')}
- 이익 성장률: {info.get('earningsGrowth', 'N/A')}

재무 건전성:
- 부채비율: {info.get('debtToEquity', 'N/A')}
- 유동비율: {info.get('currentRatio', 'N/A')}
- 잉여현금흐름: ${info.get('freeCashflow', 0):,.0f}

시장:
- 시가총액: ${info.get('marketCap', 0):,.0f}
- 52주 범위: ${info.get('fiftyTwoWeekLow', 0):.2f} ~ ${info.get('fiftyTwoWeekHigh', 0):.2f}
- 배당수익률: {info.get('dividendYield', 'N/A')}
- 애널리스트 목표가: ${info.get('targetMeanPrice', 'N/A')}
"""

print(fundamental_context)


종목: Apple Inc. (AAPL)
섹터: Technology / Consumer Electronics

가치 평가:
- PER: 32.804813
- Forward PER: 27.79463
- PBR: 43.152714
- EV/EBITDA: 25.003

수익성:
- ROE: 1.5202099
- ROA: 0.24377
- 영업이익률: 0.35374
- 순이익률: 0.27037

성장성:
- 매출 성장률: 0.157
- 이익 성장률: 0.183

재무 건전성:
- 부채비율: 102.63
- 유동비율: 0.974
- 잉여현금흐름: $106,312,753,152

시장:
- 시가총액: $3,804,263,874,560
- 52주 범위: $189.81 ~ $288.62
- 배당수익률: 0.4
- 애널리스트 목표가: $296.3305



In [19]:
# Gemini에 펀더멘탈 데이터를 넣어서 분석 요청
prompt = f"""
당신은 전문 주식 애널리스트입니다. 아래 펀더멘탈 데이터를 기반으로 이 종목의 투자 매력도를 분석해주세요.

분석 항목:
1. 밸류에이션 (PER/PBR 기준 고평가/저평가 판단)
2. 수익성 (ROE/영업이익률 기반 경쟁력)
3. 성장성 (매출/이익 성장 추세)
4. 재무 안정성 (부채비율/현금흐름)
5. 종합 의견 (매수/보유/매도 중 하나 + 근거)

한국어로 간결하게 분석해주세요.

{fundamental_context}
"""

response = model.generate_content(prompt)
print(response.text)

## Apple Inc. (AAPL) 투자 매력도 분석

전문 주식 애널리스트로서 Apple Inc.의 펀더멘탈 데이터를 기반으로 투자 매력도를 분석합니다.

---

### 1. 밸류에이션 (Valuation)

*   **PER 32.80, PBR 43.15:** 전통적인 가치 평가 지표인 PER과 PBR은 시장 평균 대비 **매우 높은 수준**입니다. 이는 Apple이 현재 시장에서 상당한 프리미엄을 받고 있으며, 미래 성장과 브랜드 가치에 대한 높은 기대치가 주가에 선반영되어 있음을 시사합니다. Forward PER(27.79)이 현재 PER보다 낮지만 여전히 높은 편이며, EV/EBITDA(25.00) 또한 높은 밸류에이션을 뒷받침합니다. 전반적으로 **고평가 영역**에 있다고 판단됩니다.

### 2. 수익성 (Profitability)

*   **ROE 1.52%, ROA 0.24%:** 제공된 데이터에 따르면 ROE와 ROA는 **이례적으로 낮은 수준**입니다. 이는 일반적으로 비효율성을 시사할 수 있으나, Apple의 사업 특성상 대규모 자사주 매입으로 인한 자본 구조 관리(자기자본 감소)로 인해 지표가 왜곡될 수 있음을 고려해야 합니다.
*   **영업이익률 35.37%, 순이익률 27.04%:** 반면, 영업이익률과 순이익률은 **매우 우수한 수준**입니다. 이는 Apple이 강력한 브랜드 파워와 제품 경쟁력을 바탕으로 높은 가격 결정력을 보유하고 있으며, 효율적인 비용 통제를 통해 탁월한 마진을 창출하고 있음을 명확히 보여줍니다. 높은 마진은 Apple의 핵심적인 경쟁 우위입니다.

### 3. 성장성 (Growth)

*   **매출 성장률 15.7%, 이익 성장률 18.3%:** 시가총액이 3조 8천억 달러에 달하는 초대형 기업임에도 불구하고, 두 자릿수의 매출 및 이익 성장률을 기록하고 있다는 점은 **매우 인상적**입니다. 이는 제품 및 서비스 전반에 걸친 지속적인 혁신과 강력한 생태계 확장 능력을 통해 꾸준히 시장 점유율을 확대

## 3. 채팅 모드 (Multi-turn Conversation)

## Apple Inc. (AAPL) 투자 매력도 분석

전문 주식 애널리스트로서 Apple Inc.의 펀더멘탈 데이터를 기반으로 투자 매력도를 분석합니다.

---

### 1. 밸류에이션 (Valuation)

*   **PER 32.80, PBR 43.15:** 전통적인 가치 평가 지표인 PER과 PBR은 시장 평균 대비 **매우 높은 수준**입니다. 이는 Apple이 현재 시장에서 상당한 프리미엄을 받고 있으며, 미래 성장과 브랜드 가치에 대한 높은 기대치가 주가에 선반영되어 있음을 시사합니다. Forward PER(27.79)이 현재 PER보다 낮지만 여전히 높은 편이며, EV/EBITDA(25.00) 또한 높은 밸류에이션을 뒷받침합니다. 전반적으로 **고평가 영역**에 있다고 판단됩니다.

### 2. 수익성 (Profitability)

*   **ROE 1.52%, ROA 0.24%:** 제공된 데이터에 따르면 ROE와 ROA는 **이례적으로 낮은 수준**입니다. 이는 일반적으로 비효율성을 시사할 수 있으나, Apple의 사업 특성상 대규모 자사주 매입으로 인한 자본 구조 관리(자기자본 감소)로 인해 지표가 왜곡될 수 있음을 고려해야 합니다.
*   **영업이익률 35.37%, 순이익률 27.04%:** 반면, 영업이익률과 순이익률은 **매우 우수한 수준**입니다. 이는 Apple이 강력한 브랜드 파워와 제품 경쟁력을 바탕으로 높은 가격 결정력을 보유하고 있으며, 효율적인 비용 통제를 통해 탁월한 마진을 창출하고 있음을 명확히 보여줍니다. 높은 마진은 Apple의 핵심적인 경쟁 우위입니다.

### 3. 성장성 (Growth)

*   **매출 성장률 15.7%, 이익 성장률 18.3%:** 시가총액이 3조 8천억 달러에 달하는 초대형 기업임에도 불구하고, 두 자릿수의 매출 및 이익 성장률을 기록하고 있다는 점은 **매우 인상적**입니다. 이는 제품 및 서비스 전반에 걸친 지속적인 혁신과 강력한 생태계 확장 능력을 통해 꾸준히 시장 점유율을 확대하고 있음을 나타냅니다. 견조한 성장세는 긍정적입니다.

### 4. 재무 안정성 (Financial Stability)

*   **부채비율 102.63%:** 부채비율은 100%를 소폭 상회하는 수준으로, 매우 공격적이라고 보기는 어렵지만 무시할 수 없는 수준입니다. 하지만 Apple과 같이 현금 흐름이 압도적인 기업의 경우 충분히 **관리 가능한 수준**으로 평가됩니다.
*   **유동비율 0.974:** 유동비율이 1.0 미만으로 단기 지급 능력에 다소 우려가 있을 수 있습니다. 그러나 이 역시 Apple의 **방대한 잉여현금흐름(FCF $106,312,753,152)**으로 충분히 상쇄됩니다. 막대한 잉여현금흐름은 Apple의 재무 안정성을 뒷받침하는 가장 강력한 요소이며, 미래 투자, 자사주 매입, 배당 등 다양한 자본 배분 전략을 가능하게 합니다. 전반적인 재무 안정성은 현금 창출력을 바탕으로 **매우 견고**하다고 판단됩니다.

---

### 5. 종합 의견

*   **종합 의견: 보유 (HOLD)**
*   **근거:** Apple은 압도적인 브랜드 가치, 강력한 생태계, 탁월한 수익성(높은 이익률), 견조한 성장성, 그리고 무엇보다 **세계 최고 수준의 잉여현금흐름**을 자랑하는 세계적인 우량 기업입니다. 이러한 펀더멘탈은 장기적인 관점에서 매우 매력적입니다.
    그러나 현재 주가는 **매우 높은 밸류에이션 영역**에 진입해 있습니다. PER 32배, PBR 43배는 현재 주가에 상당한 미래 성장 기대치를 반영하고 있으며, 52주 범위의 최고점에 근접한 가격과 애널리스트 목표가와의 제한적인 상승 여력 또한 단기적인 추가 상승 동력을 제약할 수 있습니다.

    따라서, Apple의 뛰어난 사업 경쟁력과 현금 창출력에도 불구하고, 현재의 높은 밸류에이션을 고려할 때 신규 투자자에게는 매수 매력이 다소 떨어지는 시점입니다. 이미 Apple 주식을 보유하고 있는 투자자라면, 장기적인 관점에서 기업의 지속적인 성장을 기대하며 **보유**하는 전략이 합리적이라고 판단합니다. 주가 조정을 통해 밸류에이션 부담이 완화된다면 매수 기회를 모색해 볼 수 있습니다.


=== 첫 번째 응답 ===
Apple Inc. (AAPL)의 펀더멘탈을 제공된 지표를 바탕으로 심층 분석해 드리겠습니다.

---

### **Apple Inc. (AAPL) 펀더멘탈 분석**

**종합 의견:**
Apple은 독보적인 브랜드 가치, 강력한 생태계, 그리고 막대한 현금 흐름을 기반으로 시장을 선도하는 기술 기업입니다. 높은 프리미엄 밸류에이션을 받고 있지만, 이는 뛰어난 수익성과 안정적인 성장성, 그리고 강력한 재무 건전성에 대한 시장의 신뢰를 반영합니다. 투자자들은 Apple의 지속적인 혁신과 서비스 부문의 성장에 주목하고 있습니다.

---

### **1. 가치 평가 (Valuation)**

*   **PER (주가수익비율): 32.80배**
*   **Forward PER (선행 주가수익비율): 27.79배**
    *   현재 PER은 기술 섹터 및 S&P 500 평균 대비 높은 수준으로, Apple이 프리미엄 기업으로 평가받고 있음을 보여줍니다. 이는 애플의 강력한 브랜드, 충성

=== 후속 질문 응답 ===
Apple Inc. (AAPL)의 가장 큰 리스크 요인 3가지를 분석해 드리겠습니다. 제공된 펀더멘탈 지표와 현재 시장 상황을 고려했습니다.

---

### **Apple (AAPL)의 가장 큰 리스크 요인 3가지**

**1. 높은 밸류에이션과 시장 기대치 (High Valuation & Market Expectations)**

*   **설명:** Apple은 현재 PER 32.8배, PBR 43.15배 등 기술 섹터 및 S&P 500 평균 대비 매우 높은 밸류에이션을 받고 있습니다. 이는 Apple의 뛰어난 수익성, 강력한 브랜드, 혁신 능력, 그리고 안정적인 현금 창출 능력에 대한 시장의 높은 기대를 반영합니다.
*   **리스크:** 이러한 높은 기대치에 미치지 못하는 실적(예: 아이폰 판매량 둔화, 서비스 부문 성장 둔화, 신제품 출시 지연 또는 실패)이 발생할 경우, 주가에 대한 프리미엄이 사라지면서 **가치 재평가(de-rating)에 따른 주가 하락 압력**이 매

In [20]:
# 채팅 세션 - 후속 질문 가능
chat = model.start_chat(history=[])

# 첫 번째 질문
r1 = chat.send_message(f'다음 종목의 펀더멘탈을 분석해줘:\n{fundamental_context}')
print('=== 첫 번째 응답 ===')
print(r1.text[:500])

# 후속 질문 (이전 맥락 유지)
r2 = chat.send_message('이 종목의 가장 큰 리스크 요인 3가지는?')
print('\n=== 후속 질문 응답 ===')
print(r2.text[:500])

=== 첫 번째 응답 ===
Apple Inc. (AAPL)의 펀더멘탈을 제공된 지표를 바탕으로 심층 분석해 드리겠습니다.

---

### **Apple Inc. (AAPL) 펀더멘탈 분석**

**종합 의견:**
Apple은 독보적인 브랜드 가치, 강력한 생태계, 그리고 막대한 현금 흐름을 기반으로 시장을 선도하는 기술 기업입니다. 높은 프리미엄 밸류에이션을 받고 있지만, 이는 뛰어난 수익성과 안정적인 성장성, 그리고 강력한 재무 건전성에 대한 시장의 신뢰를 반영합니다. 투자자들은 Apple의 지속적인 혁신과 서비스 부문의 성장에 주목하고 있습니다.

---

### **1. 가치 평가 (Valuation)**

*   **PER (주가수익비율): 32.80배**
*   **Forward PER (선행 주가수익비율): 27.79배**
    *   현재 PER은 기술 섹터 및 S&P 500 평균 대비 높은 수준으로, Apple이 프리미엄 기업으로 평가받고 있음을 보여줍니다. 이는 애플의 강력한 브랜드, 충성

=== 후속 질문 응답 ===
Apple Inc. (AAPL)의 가장 큰 리스크 요인 3가지를 분석해 드리겠습니다. 제공된 펀더멘탈 지표와 현재 시장 상황을 고려했습니다.

---

### **Apple (AAPL)의 가장 큰 리스크 요인 3가지**

**1. 높은 밸류에이션과 시장 기대치 (High Valuation & Market Expectations)**

*   **설명:** Apple은 현재 PER 32.8배, PBR 43.15배 등 기술 섹터 및 S&P 500 평균 대비 매우 높은 밸류에이션을 받고 있습니다. 이는 Apple의 뛰어난 수익성, 강력한 브랜드, 혁신 능력, 그리고 안정적인 현금 창출 능력에 대한 시장의 높은 기대를 반영합니다.
*   **리스크:** 이러한 높은 기대치에 미치지 못하는 실적(예: 아이폰 판매량 둔화, 서비스 부문 성장 둔화, 신제품 출시 지연 또는 실패)이 발생할 경우, 주가에 대한 프리미엄이 사라지면서 **

## 4. 우리 프로젝트에서의 실제 사용 (trading-monitor/app/api/ai/chat/route.ts)

```typescript
// Gemini 호출 부분 (실제 코드)
async function callGemini(prompt: string): Promise<string> {
  const key = process.env.GEMINI_API_KEY;
  const res = await fetch(
    `https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key=${key}`,
    {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({
        contents: [{ parts: [{ text: prompt }] }],
      }),
    }
  );
  const data = await res.json();
  return data.candidates?.[0]?.content?.parts?.[0]?.text || '';
}
```

### 환경 변수
```
# trading-monitor/.env
GEMINI_API_KEY=your_key_here
```

## 5. 요약

| 기능 | 메서드 | 용도 |
|------|--------|------|
| 텍스트 생성 | `model.generate_content()` | 단일 분석 요청 |
| 채팅 | `model.start_chat()` | 후속 질문, 맥락 유지 |
| 스트리밍 | `model.generate_content(stream=True)` | 실시간 응답 표시 |
| 이미지 분석 | `model.generate_content([image, prompt])` | 차트 이미지 분석 |

### 우리 프로젝트 적용:
- **현재**: REST API로 직접 호출 (TypeScript)
- **개선**: Python SDK로 더 풍부한 기능 활용 가능
- **펀더멘탈 연동**: yfinance 데이터 → 프롬프트에 주입 → AI 분석